# Week 05 Monday — The Jupyter Environment & NumPy Foundations

**Objective:** This notebook covers the Jupyter notebook environment, common pitfalls, and core NumPy operations including array creation, indexing, broadcasting, aggregation, and reshaping. Every section includes a markdown explanation, executable code, and an interpretation of the result — the structure expected for this week's graded EDA deliverable.

**Kata parts covered:** A (Jupyter basics + hidden-state bug), B (NumPy hands-on), C (in-notebook lookup + final assembly).

In [31]:
import numpy as np
import sys

---
## Part A — Jupyter Basics

### Kata 2: Markdown + Code Cells, Used Deliberately

A notebook alternates between **code cells** (Python sent to the kernel) and **markdown cells** (formatted text that renders in place). The narrative shape — state the question in markdown, show the check in code, interpret the result in markdown — is what turns a notebook into a readable report rather than a scratchpad.

**Question:** What is the 37th Fibonacci number, and is it prime?

In [19]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

fib_37 = fibonacci(37)
prime_check = is_prime(fib_37)
print(f"Fibonacci(37) = {fib_37}")
print(f"Is prime? {prime_check}")

Fibonacci(37) = 24157817
Is prime? False


**Answer:** The 37th Fibonacci number is **24157817**, and it **is not prime** (it factors as 139 × 173813). This confirms that Fibonacci numbers grow fast enough that primality becomes rare — a pattern worth noting if this data appeared in a real EDA.

### Kata 3: The Hidden-State Bug — Reproduced and Fixed

A Jupyter kernel holds **all variables in memory**, regardless of cell order. This means you can run cells out of order and get correct-looking results — until you restart the kernel, at which point everything breaks silently.

**How to reproduce (manually in JupyterLab):**
1. Create cell B first, containing `print(my_var)`. Run it — NameError.
2. Create cell A below it, containing `my_var = 42`. Run it.
3. Go back to cell B and run it again — it prints `42`. Appears to work.
4. **Restart Kernel and Run All** — cell B runs before cell A, and it fails again.

The fix is always the same: **define before you use, then verify with Restart Kernel + Run All.**

In [20]:
# CORRECT ORDER: definition comes BEFORE use.
# On Restart Kernel + Run All, this cell runs first, so my_var exists.
my_var = 42

In [21]:
# Now my_var is guaranteed to exist, regardless of manual run order.
print(f"my_var = {my_var}")

my_var = 42


**Observation:** The variable is accessible across cells because the kernel's memory persists, not because of anything about the file. The only reliable way to test a notebook is **Restart Kernel and Run All** — top to bottom, zero manual intervention. If that fails, the notebook was never actually correct.

---
## Part B — NumPy, Hands-On

### Kata 4: Array Creation and Inspection

NumPy arrays (`ndarray`) store elements of one fixed type contiguously in memory — the same layout a C array uses. This is why vectorized operations on arrays are dramatically faster than element-by-element Python loops.

In [22]:
# --- np.array: build from a Python list ---
a_list = np.array([1, 2, 3, 4, 5])
print("np.array([1,2,3,4,5])")
print(f"  shape: {a_list.shape}   dtype: {a_list.dtype}   size: {a_list.size}")

# --- np.zeros: pre-allocated array of zeros ---
a_zeros = np.zeros((3, 4))
print("\nnp.zeros((3, 4))")
print(f"  shape: {a_zeros.shape}   dtype: {a_zeros.dtype}   size: {a_zeros.size}")

# --- np.arange: like range() but returns an array ---
a_arange = np.arange(0, 10, 2)
print("\nnp.arange(0, 10, 2)")
print(f"  shape: {a_arange.shape}   dtype: {a_arange.dtype}   size: {a_arange.size}")

# --- np.linspace: evenly spaced values over an interval ---
a_linspace = np.linspace(0, 1, 5)
print("\nnp.linspace(0, 1, 5)")
print(f"  shape: {a_linspace.shape}   dtype: {a_linspace.dtype}   size: {a_linspace.size}")

# Quick print of actual values
print("\n--- values ---")
print(f"  a_list:    {a_list}")
print(f"  a_zeros:\n{a_zeros}")
print(f"  a_arange:  {a_arange}")
print(f"  a_linspace:{a_linspace}")

np.array([1,2,3,4,5])
  shape: (5,)   dtype: int64   size: 5

np.zeros((3, 4))
  shape: (3, 4)   dtype: float64   size: 12

np.arange(0, 10, 2)
  shape: (5,)   dtype: int64   size: 5

np.linspace(0, 1, 5)
  shape: (5,)   dtype: float64   size: 5

--- values ---
  a_list:    [1 2 3 4 5]
  a_zeros:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
  a_arange:  [0 2 4 6 8]
  a_linspace:[0.   0.25 0.5  0.75 1.  ]


**What to notice:**
- `np.zeros` defaults to `float64` — the default dtype for float data.
- `np.arange(0, 10, 2)` produces `int64` (whole numbers, no decimals).
- `np.linspace(0, 1, 5)` produces `float64` — it always includes the endpoint, unlike `arange`.
- `.shape` is a tuple: 1D arrays return `(n,)`, 2D return `(rows, cols)`.

### Kata 5: Proving the Speed Difference — List vs. Array

NumPy's vectorized multiplication runs as a compiled C loop outside the Python interpreter. A list comprehension runs one `MULTIPLY` bytecode instruction per element, inside the interpreter loop. The difference scales with data size.

In [23]:
# Build a Python list and a NumPy array, each with 1,000,000 numbers
py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

# List comprehension: one Python bytecode multiply per element
print("List comprehension [x * 2 for x in py_list]:")
%timeit [x * 2 for x in py_list]

print("\nNumPy vectorized np_arr * 2:")
%timeit np_arr * 2

List comprehension [x * 2 for x in py_list]:
63.3 ms ± 849 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)

NumPy vectorized np_arr * 2:
991 μs ± 22.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**Observation:** The NumPy version is typically **30–100x faster** than the list comprehension. This is the concrete payoff of contiguous memory + compiled loops vs. Python's per-element interpreter overhead. As dataset size grows, this gap becomes the difference between interactive exploration and waiting.

### Kata 6: Indexing and Slicing on a 2D Array

2D array indexing extends 1D: `arr[row, col]`. Slicing with `:` means 'all along that axis'. **Boolean masking** (`arr[arr > 0.5]`) replaces manual filtering loops with a single vectorized expression.

In [24]:
np.random.seed(42)  # reproducible
data = np.random.rand(10, 4)  # 10 rows, 4 columns
print("Dataset shape:", data.shape)
print(data)

Dataset shape: (10, 4)
[[0.37454012 0.95071431 0.73199394 0.59865848]
 [0.15601864 0.15599452 0.05808361 0.86617615]
 [0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451]
 [0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184]
 [0.45606998 0.78517596 0.19967378 0.51423444]
 [0.59241457 0.04645041 0.60754485 0.17052412]
 [0.06505159 0.94888554 0.96563203 0.80839735]
 [0.30461377 0.09767211 0.68423303 0.44015249]]


In [25]:
# Single row (row 3)
print("Row 3:", data[3])

# Single column (column 2 — every row, column 2)
print("Column 2:", data[:, 2])

# Sub-block: rows 2–5, columns 1–2
print("\nSub-block (rows 2–5, cols 1–2):")
print(data[2:5, 1:3])

# Boolean mask: every value > 0.5
mask = data > 0.5
print(f"\nValues > 0.5 ({mask.sum()} of {data.size}):", data[mask])

Row 3: [0.83244264 0.21233911 0.18182497 0.18340451]
Column 2: [0.73199394 0.05808361 0.02058449 0.18182497 0.43194502 0.29214465
 0.19967378 0.60754485 0.96563203 0.68423303]

Sub-block (rows 2–5, cols 1–2):
[[0.70807258 0.02058449]
 [0.21233911 0.18182497]
 [0.52475643 0.43194502]]

Values > 0.5 (18 of 40): [0.95071431 0.73199394 0.59865848 0.86617615 0.60111501 0.70807258
 0.96990985 0.83244264 0.52475643 0.61185289 0.78517596 0.51423444
 0.59241457 0.60754485 0.94888554 0.96563203 0.80839735 0.68423303]


**What just happened with boolean masking:**
1. `data > 0.5` builds a same-shaped `bool` array (`True` where the condition holds).
2. `data[mask]` returns a **1D array** of only the `True` elements.

This replaces the `for` loop + `if` pattern from plain Python with one expression — and it runs in compiled C, not Python bytecode.

### Kata 7: Broadcasting — Column Mean-Centering

Broadcasting lets you combine arrays of different shapes without writing a loop. The rule: compare dimensions trailing-edge-first; two dimensions are compatible when they are equal **or** one is exactly `1`.

**Real use case:** Subtracting each column's mean from every row in that column — a standard preprocessing step (mean-centering) used before PCA, clustering, and regression later this week.

In [26]:
# Step 1: compute the mean of each column (result is shape (4,))
col_means = data.mean(axis=0)
print("Column means:", col_means)
print("Shape of col_means:", col_means.shape)

# Step 2: subtract from every row using broadcasting
# data is (10, 4), col_means is (4,) → broadcasts to (10, 4) by matching the last axis
data_centered = data - col_means

# Verify: the mean of each column in the centered data should be ~0
print("\nColumn means after centering (should be ~0):")
print(data_centered.mean(axis=0))

Column means: [0.42983615 0.45695548 0.41736604 0.52090484]
Shape of col_means: (4,)

Column means after centering (should be ~0):
[-6.66133815e-17 -2.22044605e-17  1.66533454e-17  4.44089210e-17]


**What happened:** `data` is shape `(10, 4)` and `col_means` is shape `(4,)`. Broadcasting aligns them on the trailing axis (size 4 matches), then virtually replicates `col_means` across all 10 rows — no copy, no loop. The resulting means are effectively zero (floating-point rounding only).

### Kata 8: Aggregating Along an Axis

`axis=0` collapses **rows** → result has one value **per column** (shape matches column count).
`axis=1` collapses **columns** → result has one value **per row** (shape matches row count).

Getting axis backward is a common mistake. The reliable fix: **check the shape of the result**.

In [27]:
mean_axis0 = data.mean(axis=0)
mean_axis1 = data.mean(axis=1)

print(f"data shape:              {data.shape}")
print(f"mean(axis=0) shape:      {mean_axis0.shape}  ← one value per column (collapsed rows)")
print(f"mean(axis=1) shape:      {mean_axis1.shape}  ← one value per row    (collapsed columns)")

print("\nmean(axis=0):", mean_axis0)
print("mean(axis=1):", mean_axis1)

data shape:              (10, 4)
mean(axis=0) shape:      (4,)  ← one value per column (collapsed rows)
mean(axis=1) shape:      (10,)  ← one value per row    (collapsed columns)

mean(axis=0): [0.42983615 0.45695548 0.41736604 0.52090484]
mean(axis=1): [0.66397671 0.30906823 0.57492048 0.35250281 0.38804321 0.35246331
 0.48878854 0.35423349 0.69699163 0.38166785]


**Why different shapes:** `axis=0` removes the row dimension, leaving `(4,)` — one statistic per column. `axis=1` removes the column dimension, leaving `(10,)` — one statistic per row. Always verify `.shape` after an aggregation.

### Kata 9: Reshape — View vs. Copy

`.reshape()` changes an array's shape **without changing its underlying data**. Crucially, it often returns a **view** — sharing the same memory as the original. Mutating the reshaped array silently mutates the original. This must be verified by running it, not assumed from memory.

In [28]:
original = np.array([1, 2, 3, 4, 5, 6])
reshaped = original.reshape(2, 3)

print("original (before mutation):", original)
print("reshaped (before mutation):")
print(reshaped)
print(f"\nAre they the same object? {original is reshaped}")
print(f"Do they share memory?    {np.shares_memory(original, reshaped)}")

# Mutate one element in the reshaped view
reshaped[0, 0] = 999

print("\n--- After setting reshaped[0,0] = 999 ---")
print("original:", original)
print("reshaped:")
print(reshaped)

print("\nObservation: modifying reshaped[0,0] changed original[0] too.")
print("This is a VIEW — same underlying memory.")
print("To get an independent copy, use .copy():")
independent = original.copy().reshape(2, 3)
independent[0, 0] = 0
print(f"After changing independent[0,0] = 0, original is still: {original}")

original (before mutation): [1 2 3 4 5 6]
reshaped (before mutation):
[[1 2 3]
 [4 5 6]]

Are they the same object? False
Do they share memory?    True

--- After setting reshaped[0,0] = 999 ---
original: [999   2   3   4   5   6]
reshaped:
[[999   2   3]
 [  4   5   6]]

Observation: modifying reshaped[0,0] changed original[0] too.
This is a VIEW — same underlying memory.
To get an independent copy, use .copy():
After changing independent[0,0] = 0, original is still: [999   2   3   4   5   6]


**Key takeaway:** `.reshape()` returns a view by default — mutations leak through to the original. Use `.copy()` when you need an independent array. This is worth memorizing: the most dangerous bugs are the ones that silently corrupt data.

---
## Part C — Bring It Together

### Kata 10: Looking Up a NumPy Function Without Leaving the Notebook

Jupyter’s built-in tools make browser-free lookup straightforward:
- **`Shift+Tab`** inside a function’s parentheses → signature + docstring tooltip.
- **Trailing `?`** after any name → full help rendered inline below the cell.
- **`help(name)`** → standard Python builtin, works universally in any environment.

Below we look up `np.percentile` — a function we haven’t used yet.

In [33]:
np.percentile?

Signature:      
np.percentile(
    a,
    q,
    axis=None,
    out=None,
    overwrite_input=False,
    method='linear',
    keepdims=False,
    *,
    weights=None,
    interpolation=None,
)
Call signature:  np.percentile(*args, **kwargs)
Type:            _ArrayFunctionDispatcher
String form:     <function percentile at 0x71c5c8738b80>
File:            ~/Documents/internship/Jupyter Environment & NumPy Foundations/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py
Docstring:      
Compute the q-th percentile of the data along the specified axis.

Returns the q-th percentile(s) of the array elements.

Parameters
----------
a : array_like of real numbers
    Input array or object that can be converted to an array.
q : array_like of float
    Percentage or sequence of percentages for the percentiles to compute.
    Values must be between 0 and 100 inclusive.
axis : {int, tuple of int, None}, optional
    Axis or axes along which the percentiles are computed. The
    de

In [29]:
help(np.percentile)

Help on _ArrayFunctionDispatcher in module numpy:

percentile(a, q, axis=None, out=None, overwrite_input=False, method='linear', keepdims=False, *, weights=None, interpolation=None)
    Compute the q-th percentile of the data along the specified axis.
    
    Returns the q-th percentile(s) of the array elements.
    
    Parameters
    ----------
    a : array_like of real numbers
        Input array or object that can be converted to an array.
    q : array_like of float
        Percentage or sequence of percentages for the percentiles to compute.
        Values must be between 0 and 100 inclusive.
    axis : {int, tuple of int, None}, optional
        Axis or axes along which the percentiles are computed. The
        default is to compute the percentile(s) along a flattened
        version of the array.
    out : ndarray, optional
        Alternative output array in which to place the result. It must
        have the same shape and buffer length as the expected output,
        but t

In [30]:
# Practical use: find the 25th, 50th (median), and 75th percentiles of column 0
percentiles = np.percentile(data[:, 0], [25, 50, 75])
print("Column 0 percentiles [25th, 50th, 75th]:", percentiles)

Column 0 percentiles [25th, 50th, 75th]: [0.30433512 0.41530505 0.5989399 ]


**What `np.percentile` does:** computes the value below which a given percentage of data falls. Useful for understanding distributions in EDA — exactly the kind of function you'd reach for during this week's graded deliverable.

---
## Notebook Quality Check

Before considering any notebook done, run **Restart Kernel and Run All** from the menu (Kernel → Restart Kernel and Run All Cells). If every cell completes without error and the outputs are consistent, the notebook is correct.

Checklist for this notebook:
- [x] Title and intro markdown cell
- [x] Imports gathered at the top
- [x] Markdown commentary before and after every code section
- [x] Hidden-state bug demonstrated and fixed (correct order survives clean run)
- [x] Arrays created four ways, `.shape`/`.dtype`/`.size` printed for each
- [x] List-vs-array speed comparison with actual `%timeit` numbers
- [x] 2D indexing, slicing, and boolean masking on a seeded dataset
- [x] Broadcasting: column mean subtraction with no manual loop
- [x] Axis aggregation with both shapes printed and explained
- [x] Reshape view-vs-copy verified by mutation, compared against prediction
- [x] `np.percentile` looked up via `help()` without leaving the notebook
- [x] Logical flow: question → code → interpretation throughout

In [32]:
print("All cells executed successfully.")
print(f"Python version: {sys.version}")
print(f"NumPy version:  {np.__version__}")

All cells executed successfully.
Python version: 3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]
NumPy version:  2.2.6
